[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataiku/kiji-inspector/blob/main/demo/home_repair/home_repair_colab.ipynb)

# Home Repair Agent Demo with SAE Activation Analysis

This notebook reproduces `demo/home_repair/home_repair_demo.py` step by step in Colab.

**What it does**
1. For each of three repair problems (dishwasher leak, stuck disposal, noisy water heater) the model is given tool results (manual, parts, tutorials, pro quote) and asked to analyze them — one LLM call per (problem, tool) pair.
2. The model then produces a final cross-problem recommendation.
3. Activations at every decision point are captured via HuggingFace forward hooks.
4. A trained Sparse Autoencoder (SAE) decomposes those activations into interpretable features.
5. The results are packaged into `ui_data.json` for the static `index.html` viewer.

**Requirements**
- Runtime → Change runtime type → **GPU (A100, High-RAM)**. The base model is `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16` (~30B params). A T4/L4 will OOM.
- A HuggingFace token. The easiest way is to add it as a Colab Secret named `HF_TOKEN`.
- (Optional) A YouTube Data API v3 key as Colab Secret `YOUTUBE_API_KEY` to fetch real tutorial results instead of mock data.
- (Optional) A path to a local pipeline output directory in `SAE_LOCAL_DIR` to use a locally trained SAE + contrastive feature themes instead of the HuggingFace Hub copy.

## 1. Install dependencies

In [ ]:
!pip install -q -U kiji-inspector transformers accelerate huggingface_hub

## 2. HuggingFace login + optional YouTube API key

`HF_TOKEN` is required. `YOUTUBE_API_KEY` is optional — if missing, the TutorialSearch tool falls back to mock results.

In [ ]:
from huggingface_hub import login

YOUTUBE_API_KEY = None
SAE_LOCAL_DIR = None  # set to a local pipeline output dir to use a trained SAE + contrastive themes

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in via Colab secret HF_TOKEN.")
    try:
        YOUTUBE_API_KEY = userdata.get("YOUTUBE_API_KEY")
        print("Loaded YOUTUBE_API_KEY from Colab secret.")
    except Exception:
        print("No YOUTUBE_API_KEY secret — TutorialSearch will use mock data.")
except Exception:
    login()

## 3. Write `home_repair.json` to disk

`build_ui_data` looks for this file alongside the script to pull contrast-type descriptions for the comparison chart. Writing it here means the rest of the notebook can stay faithful to the original script.

In [ ]:
import json
from pathlib import Path

HOME_REPAIR_CONFIG = {
    "name": "home_repair",
    "system_prompt": (
        "You are a home repair advisor helping homeowners diagnose appliance and household problems. "
        "You analyze diagnostic data from various tools to determine whether a problem requires "
        "professional service or can be safely handled as a DIY repair. Always consider safety first, "
        "then cost, difficulty, and warranty implications. Ground your advice in specific details "
        "from the data provided."
    ),
    "tools": [
        {"name": "manual_check", "description": "Look up appliance troubleshooting guides, error codes, manufacturer-recommended diagnostic steps, and safety warnings"},
        {"name": "parts_search", "description": "Search for replacement parts with pricing, availability, compatibility information, and estimated shipping times"},
        {"name": "tutorial_search", "description": "Find video tutorials and step-by-step repair guides with difficulty ratings, required tools, and estimated completion times"},
        {"name": "pro_quote", "description": "Get professional repair service quotes including labor costs, typical turnaround times, service warranties, and urgency assessment"},
    ],
    "contrast_types": {
        "diy_vs_professional": "One repair is safely DIY-able with basic household tools and minimal experience, while the other involves safety hazards (gas, high voltage, structural) or specialized equipment that requires a licensed professional. Example: replacing a dishwasher door gasket (DIY) vs servicing a gas water heater burner assembly (professional).",
        "urgent_vs_planned": "One problem needs immediate attention due to active damage or safety risk (water leak, gas smell, electrical hazard), while the other is a nuisance that can be scheduled at convenience. Example: a burst pipe flooding the kitchen (urgent) vs a garbage disposal that jams occasionally (planned).",
        "cheap_fix_vs_replacement": "One problem is a simple inexpensive part swap under $50, while the other suggests the appliance is near end-of-life and full replacement is more cost-effective than repair. Example: replacing a $13 spray arm seal (cheap fix) vs repairing a 15-year-old water heater with a corroding tank (replacement).",
        "safe_vs_hazardous": "One repair poses no meaningful risk to a careful homeowner, while the other involves gas lines, live electrical panels, pressurized systems, or structural load-bearing elements. Example: unclogging a garbage disposal with an Allen wrench (safe) vs troubleshooting a gas water heater pilot assembly (hazardous).",
        "warranty_covered_vs_out_of_pocket": "One appliance is under manufacturer or extended warranty making professional service free or low-cost, while the other has expired coverage making DIY the economical choice. Example: a 1-year-old dishwasher under warranty (covered) vs a 9-year-old water heater past its 6-year warranty (out of pocket).",
    },
}

DEMO_DIR = Path("/content/home_repair")
DEMO_DIR.mkdir(parents=True, exist_ok=True)
HOME_REPAIR_JSON = DEMO_DIR / "home_repair.json"
HOME_REPAIR_JSON.write_text(json.dumps(HOME_REPAIR_CONFIG, indent=2))
print(f"Wrote {HOME_REPAIR_JSON}")

## 4. Imports and constants

In [ ]:
from __future__ import annotations

import gc
import textwrap

import numpy as np
import torch

from kiji_inspector.core.sae import SAE

_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
_SAE_REPO_ID = "davidnet/kiji-inspector-NVIDIA-Nemotron-3-Nano-30B-A3B-BF16-Home-scenarios"
_SAE_LAYER = 20

# Transformer layers to hook for activation extraction (must include _SAE_LAYER).
_HOOK_LAYERS = [10, 20, 30, 40, 50]

_SYSTEM_PROMPT = (
    "You are an experienced home repair advisor. You help homeowners diagnose "
    "appliance problems and decide whether to attempt a DIY repair or hire a "
    "professional. Always consider safety first, then cost, difficulty, and "
    "warranty implications. Reference specific details from the data provided."
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

## 5. Define the three repair problems

In [ ]:
_PROBLEMS = [
    {
        "id": "dishwasher_leak",
        "summary": "Dishwasher leaking water from the bottom",
        "appliance": "Bosch 500 Series dishwasher",
        "age": "3 years",
        "details": (
            "Water pools under the front of the unit about 15 minutes into "
            "the wash cycle. No error codes on the display."
        ),
    },
    {
        "id": "disposal_stuck",
        "summary": "Garbage disposal hums but won't spin",
        "appliance": "InSinkErator Badger 5",
        "age": "2 years",
        "details": (
            "Motor hums when the switch is flipped but blades don't turn. "
            "Was working fine yesterday. No unusual smell."
        ),
    },
    {
        "id": "water_heater_noise",
        "summary": "Water heater making loud popping and rumbling sounds",
        "appliance": "Rheem 50-gallon gas water heater",
        "age": "9 years",
        "details": (
            "Loud popping when heating up, especially in the morning. "
            "Hot water takes longer to reach faucets. Slight rust tinge "
            "in the first few seconds of hot water."
        ),
    },
]

## 6. Mock tool data + optional YouTube API

Four "tools" return canned JSON: a manual lookup, parts catalog, tutorial search, and a professional quote. If `YOUTUBE_API_KEY` is set, TutorialSearch hits the real YouTube Data API v3 instead.

In [ ]:
_MANUAL_DATA = {
    "dishwasher_leak": {
        "model": "Bosch 500 Series SHP65T55UC",
        "possible_causes": [
            "Door gasket worn or cracked -- inspect for debris buildup",
            "Pump seal failure (common at 2-3 years of heavy use)",
            "Water inlet valve connections loose or corroded",
            "Spray arm seal worn -- water escapes during rotation",
        ],
        "diy_difficulty": "Moderate",
        "safety": "Disconnect power at breaker and shut water supply valve before inspection.",
        "tools_needed": ["Phillips screwdriver", "Torx T20 driver", "flashlight", "towels"],
    },
    "disposal_stuck": {
        "model": "InSinkErator Badger 5",
        "possible_causes": [
            "Jammed flywheel -- foreign object wedged between impellers",
            "Thermal overload tripped (red reset button on bottom of unit)",
            "Capacitor failure -- motor hums but cannot start rotation (less common)",
        ],
        "diy_difficulty": "Easy",
        "safety": "NEVER put hand inside disposal. Ensure power is OFF at breaker before clearing a jam.",
        "tools_needed": ["1/4-inch Allen wrench (hex key)", "flashlight", "tongs or pliers"],
        "quick_fix": (
            "Insert 1/4-inch Allen wrench into the hex socket on the bottom center "
            "of the unit. Rotate back and forth to free the jam. Remove debris with "
            "tongs. Press the red reset button. Restore power and test."
        ),
    },
    "water_heater_noise": {
        "model": "Rheem Performance 50-Gal Gas XR50T06EC36U1",
        "possible_causes": [
            "Sediment buildup on tank bottom -- mineral deposits from hard water heat and pop",
            "Anode rod depleted -- sacrificial rod no longer protecting tank lining",
            "Scale buildup on burner assembly reducing heat transfer efficiency",
            "Possible tank corrosion if popping is severe and rust is present in water",
        ],
        "diy_difficulty": "Moderate to Difficult",
        "safety": (
            "GAS APPLIANCE: Risk of scalding burns from hot water and gas leaks if "
            "connections are disturbed. Turn off gas supply valve before any work. "
            "If you smell gas, leave immediately and call your gas utility."
        ),
        "tools_needed": [
            "Garden hose",
            "1-1/16 inch anode rod socket",
            "pipe wrench",
            "Teflon tape",
        ],
    },
}

_PARTS_DATA = {
    "dishwasher_leak": {
        "parts": [
            {"name": "Door Gasket Seal (OEM)", "part_no": "00744367", "price": 42.99, "in_stock": True},
            {"name": "Drain Pump Assembly", "part_no": "00631200", "price": 89.50, "in_stock": True},
            {"name": "Water Inlet Valve", "part_no": "00622058", "price": 55.75, "in_stock": True},
            {"name": "Spray Arm Seal Kit", "part_no": "00165259", "price": 12.99, "in_stock": True},
        ],
        "diy_cost_range": "$13 - $90 depending on which part has failed",
    },
    "disposal_stuck": {
        "parts": [
            {"name": "Self-Service Wrench Kit", "part_no": "WRN-00", "price": 7.99, "in_stock": True},
            {"name": "Badger 5 Replacement Unit (if motor failed)", "part_no": "?", "price": 99.00, "in_stock": True},
        ],
        "diy_cost_range": "$0 - $8 if jam clears; $99 + install if motor is dead",
    },
    "water_heater_noise": {
        "parts": [
            {"name": "Aluminum Anode Rod (Rheem-compatible)", "part_no": "SP11526", "price": 29.99, "in_stock": True},
            {"name": "Tank Flush Kit (hose + valve adapter)", "part_no": "FK-100", "price": 14.99, "in_stock": True},
            {"name": "Drain Valve Replacement", "part_no": "SP12112", "price": 11.49, "in_stock": True},
            {"name": "Rheem 50-Gal Replacement Unit (if tank is corroded)", "part_no": "XG50T06EC36U1", "price": 649.00, "in_stock": True},
        ],
        "diy_cost_range": "$15 - $45 for maintenance parts; $649+ if tank replacement needed",
    },
}

_TUTORIAL_DATA = {
    "dishwasher_leak": {
        "source": "mock",
        "results": [
            {"title": "How to Fix a Leaking Dishwasher - 5 Most Common Causes", "channel": "RepairClinic", "views": "1.2M", "duration": "12:34", "difficulty": "Beginner-Intermediate"},
            {"title": "Bosch Dishwasher Door Gasket Replacement", "channel": "AppliancePartsPros", "views": "340K", "duration": "8:15", "difficulty": "Beginner"},
            {"title": "Dishwasher Pump Seal: When to Replace vs Repair", "channel": "FixItHome", "views": "89K", "duration": "15:02", "difficulty": "Intermediate"},
        ],
    },
    "disposal_stuck": {
        "source": "mock",
        "results": [
            {"title": "Garbage Disposal Humming But Not Working? Easy Fix!", "channel": "HomeRepairTutor", "views": "2.8M", "duration": "4:22", "difficulty": "Beginner"},
            {"title": "How to Unjam a Garbage Disposal in 60 Seconds", "channel": "ThisOldHouse", "views": "1.5M", "duration": "3:10", "difficulty": "Beginner"},
            {"title": "InSinkErator Reset Button and Allen Wrench Fix", "channel": "DIYWithMike", "views": "620K", "duration": "5:45", "difficulty": "Beginner"},
        ],
    },
    "water_heater_noise": {
        "source": "mock",
        "results": [
            {"title": "Water Heater Making Noise? Here's Why and How to Fix It", "channel": "RogerWakefield", "views": "890K", "duration": "18:30", "difficulty": "Intermediate-Advanced"},
            {"title": "How to Flush a Water Heater (Step by Step)", "channel": "ThisOldHouse", "views": "3.1M", "duration": "10:15", "difficulty": "Intermediate"},
            {"title": "Replacing a Water Heater Anode Rod - Is It Worth It?", "channel": "TechDIY", "views": "450K", "duration": "14:20", "difficulty": "Intermediate"},
        ],
    },
}

_PRO_QUOTE_DATA = {
    "dishwasher_leak": {
        "diagnosis_fee": 89,
        "repair_estimates": [
            {"repair": "Door gasket replacement", "labor": 120, "parts": 43, "total": 163, "time": "1 hour"},
            {"repair": "Pump seal replacement", "labor": 180, "parts": 90, "total": 270, "time": "1.5 hours"},
            {"repair": "Inlet valve replacement", "labor": 150, "parts": 56, "total": 206, "time": "1 hour"},
        ],
        "warranty_on_repair": "90-day parts and labor",
        "urgency": "Moderate -- continued use risks water damage to flooring",
        "next_available": "2-3 business days",
    },
    "disposal_stuck": {
        "diagnosis_fee": 75,
        "repair_estimates": [
            {"repair": "Clear jam + reset", "labor": 75, "parts": 0, "total": 75, "time": "30 min"},
            {"repair": "Full unit replacement (Badger 5)", "labor": 150, "parts": 99, "total": 249, "time": "1.5 hours"},
        ],
        "warranty_on_repair": "90-day labor, manufacturer warranty on new unit",
        "urgency": "Low -- disposal is non-essential; sink still drains",
        "next_available": "3-5 business days",
    },
    "water_heater_noise": {
        "diagnosis_fee": 95,
        "repair_estimates": [
            {"repair": "Tank flush + anode rod replacement", "labor": 200, "parts": 45, "total": 245, "time": "2 hours"},
            {"repair": "Full unit replacement (50-gal gas)", "labor": 450, "parts": 649, "total": 1099, "time": "4-6 hours"},
        ],
        "warranty_on_repair": "1-year labor, 6-year tank on new unit",
        "urgency": "Moderate-High -- sediment reduces efficiency; tank corrosion risk increases with age",
        "next_available": "1-2 business days (prioritized for gas appliances)",
    },
}

_TOOLS = {
    "ManualCheck": (_MANUAL_DATA, "appliance troubleshooting guide"),
    "PartsSearch": (_PARTS_DATA, "replacement parts and pricing"),
    "TutorialSearch": (_TUTORIAL_DATA, "repair video tutorials"),
    "ProQuote": (_PRO_QUOTE_DATA, "professional repair quotes"),
}


def _fetch_youtube_tutorials(query: str, api_key: str) -> dict:
    """Search YouTube Data API v3 for repair tutorials (real API)."""
    import urllib.parse
    import urllib.request

    params = urllib.parse.urlencode({
        "part": "snippet",
        "q": query,
        "type": "video",
        "maxResults": 3,
        "key": api_key,
    })
    url = f"https://www.googleapis.com/youtube/v3/search?{params}"
    with urllib.request.urlopen(url, timeout=10) as resp:
        data = json.loads(resp.read())

    return {
        "source": "youtube_api",
        "results": [
            {
                "title": item["snippet"]["title"],
                "channel": item["snippet"]["channelTitle"],
                "video_id": item["id"]["videoId"],
                "url": f"https://youtube.com/watch?v={item['id']['videoId']}",
                "description": item["snippet"]["description"][:200],
            }
            for item in data.get("items", [])
            if item.get("id", {}).get("videoId")
        ],
    }


def _get_tool_result(tool_name: str, problem_id: str, youtube_api_key: str | None = None) -> str:
    """Return tool result as JSON. Uses real YouTube API if a key is provided."""
    if tool_name == "TutorialSearch" and youtube_api_key:
        problem = next(p for p in _PROBLEMS if p["id"] == problem_id)
        query = f"{problem['appliance']} {problem['summary']} repair tutorial"
        try:
            result = _fetch_youtube_tutorials(query, youtube_api_key)
            print(f"    (YouTube API: {len(result['results'])} results)")
            return json.dumps(result, indent=2)
        except Exception as e:
            print(f"    YouTube API failed ({e}), using mock data")

    source = _TOOLS[tool_name][0]
    data = source.get(problem_id, {"error": f"No data for '{problem_id}'"})
    return json.dumps(data, indent=2)

## 7. HFEngine: a single HuggingFace model for generation **and** activation extraction

Generation uses `model.generate()`. Activation extraction runs a separate forward pass through the transformer body with forward hooks registered on the target layers. Hooks are only active during extraction, not during generation.

In [ ]:
class HFEngine:
    def __init__(
        self,
        model_name: str = _MODEL_NAME,
        device: str = "auto",
        dtype: str = "bfloat16",
        max_new_tokens: int = 400,
    ):
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.max_new_tokens = max_new_tokens
        self.prompt_log: list[tuple[str, str]] = []

        torch_dtype = getattr(torch, dtype)

        print(f"  Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        load_kwargs: dict = {"torch_dtype": torch_dtype}
        if device == "auto":
            load_kwargs["device_map"] = "auto"
        else:
            load_kwargs["device_map"] = {"": device}

        self.model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
        self.model.eval()

        self._input_device = self._find_input_device()
        self.hidden_size = (
            getattr(self.model.config, "hidden_size", None)
            or getattr(self.model.config, "text_config", self.model.config).hidden_size
        )

        print(f"  Model ready on {self._input_device} ({torch_dtype})")
        print(f"  hidden_size: {self.hidden_size}")

    def _find_input_device(self) -> torch.device:
        for attr_path in (
            "language_model.model.embed_tokens",
            "language_model.embed_tokens",
            "model.embed_tokens",
            "transformer.wte",
        ):
            obj = self.model
            try:
                for part in attr_path.split("."):
                    obj = getattr(obj, part)
                return next(obj.parameters()).device
            except (AttributeError, StopIteration):
                continue
        return next(self.model.parameters()).device

    def _get_model_layers(self):
        if hasattr(self.model, "language_model"):
            lm = self.model.language_model
            if hasattr(lm, "model") and hasattr(lm.model, "layers"):
                return lm.model.layers
            if hasattr(lm, "layers"):
                return lm.layers
        if hasattr(self.model, "model") and hasattr(self.model.model, "layers"):
            return self.model.model.layers
        raise AttributeError(f"Cannot locate transformer layers for {type(self.model).__name__}")

    def _get_inner_model(self):
        if hasattr(self.model, "language_model"):
            lm = self.model.language_model
            if hasattr(lm, "model"):
                return lm.model
            return lm
        if hasattr(self.model, "model"):
            return self.model.model
        return self.model

    def _build_prompt(self, system: str, user: str) -> str:
        try:
            messages = [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ]
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            messages = [{"role": "user", "content": f"{system}\n\n{user}"}]
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )

    def generate(self, prompt: str, step_label: str, max_tokens: int | None = None) -> str:
        self.prompt_log.append((step_label, prompt))

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self._input_device)
        prompt_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_tokens or self.max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
            )

        new_tokens = output_ids[0][prompt_len:]
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        print(f"  [{step_label}] Generated {len(new_tokens)} tokens")
        return response

    def extract_all_prompts(self, layers: list[int]) -> list[tuple[str, dict[str, np.ndarray]]]:
        model_layers = self._get_model_layers()
        activations: dict[str, torch.Tensor] = {}
        hooks = []

        def _make_hook(name: str):
            def hook(module, input, output):
                act = output[0] if isinstance(output, tuple) else output
                activations[name] = act.detach().cpu().to(torch.float32)
            return hook

        for idx in layers:
            if idx < len(model_layers):
                h = model_layers[idx].register_forward_hook(_make_hook(f"residual_{idx}"))
                hooks.append(h)

        inner = self._get_inner_model()
        results: list[tuple[str, dict[str, np.ndarray]]] = []

        for step_label, prompt in self.prompt_log:
            activations.clear()
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self._input_device)
            with torch.no_grad():
                inner(**inputs)

            result = {name: act[:, -1, :].squeeze(0).numpy() for name, act in activations.items()}
            results.append((step_label, result))
            print(f"    {step_label}: {len(result)} layers extracted")

        for h in hooks:
            h.remove()

        return results

    def cleanup(self):
        if hasattr(self, "model") and self.model is not None:
            del self.model
            self.model = None
        if hasattr(self, "tokenizer") and self.tokenizer is not None:
            del self.tokenizer
            self.tokenizer = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("  Model released.")

## 8a. Start the live progress viewer

Starts a background static HTTP server and opens the static viewer in a new tab. Defines `write_progress(...)` so downstream cells can push status updates.

The viewer polls `output/progress.json` every couple of seconds (live signal) and `output/ui_data.json` (final results). Run this cell **before** Phase 1 so you can watch the agent's progress as each step runs.

Re-running is safe: the server is only started once.

In [ ]:
import json
import shutil
import socket
import subprocess
import urllib.request
from pathlib import Path

from google.colab import output

OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SERVE_ROOT = Path("/content/serve")
SERVE_ROOT.mkdir(parents=True, exist_ok=True)

# Symlink so any later write to OUTPUT_DIR is immediately served.
serve_output = SERVE_ROOT / "output"
if serve_output.is_symlink():
    serve_output.unlink()
elif serve_output.exists():
    shutil.rmtree(serve_output)
serve_output.symlink_to(OUTPUT_DIR)

# Fetch the static viewer (overwrites any prior copy).
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/dataiku/kiji-inspector/main/demo/home_repair/index.html",
    SERVE_ROOT / "index.html",
)

PROGRESS_PATH = OUTPUT_DIR / "progress.json"


def write_progress(phase: str, current: int, total: int, label: str) -> None:
    """Push a live progress update for the static viewer to pick up on its next poll."""
    payload = {"phase": phase, "current": current, "total": total, "label": label}
    tmp = PROGRESS_PATH.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(payload))
    tmp.replace(PROGRESS_PATH)


write_progress("waiting", 0, 0, "Waiting for Phase 1 to start...")

# Start a static HTTP server (idempotent).
PORT = 8000


def _port_in_use(port: int) -> bool:
    with socket.socket() as s:
        try:
            s.bind(("127.0.0.1", port))
            return False
        except OSError:
            return True


if not _port_in_use(PORT):
    subprocess.Popen(
        ["python", "-m", "http.server", str(PORT), "--directory", str(SERVE_ROOT)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print(f"Started static server on port {PORT}")
else:
    print(f"Server already running on port {PORT}")

output.serve_kernel_port_as_window(PORT)

## 8. Phase 1 — Load the model and run the multi-step home repair analysis

Downloads the 30B model and runs 13 LLM calls (3 problems × 4 tools + 1 final recommendation). Expect ~10-20 minutes on an A100.

In [ ]:
def run_home_repair_analysis(
    engine: HFEngine, youtube_api_key: str | None = None
) -> tuple[str, dict[str, str]]:
    per_problem_analyses: dict[str, str] = {}
    all_context = ""

    total_steps = len(_PROBLEMS) * len(_TOOLS) + 1
    step_idx = 0
    write_progress("phase1", 0, total_steps, "Phase 1 starting...")

    for problem in _PROBLEMS:
        pid = problem["id"]
        print(f"\n  --- Analyzing: {problem['summary']} ---")
        problem_context = ""

        for tool_name, (_, tool_desc) in _TOOLS.items():
            tool_result = _get_tool_result(tool_name, pid, youtube_api_key)
            step_label = f"{pid}_{tool_name}"

            user_msg = (
                f"A homeowner needs help with: {problem['summary']}\n"
                f"Appliance: {problem['appliance']} ({problem['age']} old)\n"
                f"Details: {problem['details']}\n\n"
            )
            if problem_context:
                user_msg += f"Your analysis so far:\n{problem_context}\n\n"
            user_msg += (
                f"Here is the {tool_desc} data:\n"
                f"```json\n{tool_result}\n```\n\n"
                f"Analyze this {tool_desc} data. Highlight key takeaways, "
                f"safety concerns, and whether this points toward DIY or "
                f"professional repair. Be specific with numbers."
            )

            step_idx += 1
            write_progress(
                "phase1", step_idx, total_steps,
                f"Phase 1 \u00b7 step {step_idx}/{total_steps} \u00b7 {pid} \u00b7 {tool_name}",
            )
            prompt = engine._build_prompt(_SYSTEM_PROMPT, user_msg)
            analysis = engine.generate(prompt, step_label, max_tokens=300)
            problem_context += f"\n[{tool_name}] {analysis.strip()}\n"

        per_problem_analyses[pid] = problem_context
        all_context += f"\n=== {problem['summary']} ===\n{problem_context}\n"

    print("\n  --- Final Recommendation ---")
    truncated_context = all_context
    if len(truncated_context) > 4000:
        truncated_context = "...(earlier analysis truncated)...\n" + truncated_context[-4000:]
    final_user_msg = (
        "You have analyzed three home repair problems. "
        "Here is a summary of your analysis:\n\n"
        f"{truncated_context}\n\n"
        "Now provide your final recommendation for each problem:\n"
        "1. DIY or hire a professional? Why?\n"
        "2. Estimated cost (DIY vs professional)\n"
        "3. Safety considerations\n"
        "4. Urgency level (fix now / schedule soon / can wait)\n"
        "5. Priority order: which problem should be addressed first?"
    )

    step_idx += 1
    write_progress(
        "phase1", step_idx, total_steps,
        f"Phase 1 \u00b7 step {step_idx}/{total_steps} \u00b7 final cross-problem recommendation",
    )
    final_prompt = engine._build_prompt(_SYSTEM_PROMPT, final_user_msg)
    final_rec = engine.generate(final_prompt, "final_recommendation", max_tokens=500)
    write_progress("phase1_done", total_steps, total_steps, "Phase 1 complete")
    return final_rec, per_problem_analyses


print("[Phase 1] Loading model...")
engine = HFEngine(device="auto", dtype="bfloat16", max_new_tokens=400)

print("\n[Phase 1] Running home repair analysis...")
final_recommendation, per_problem = run_home_repair_analysis(engine, YOUTUBE_API_KEY)

print("\n" + "=" * 60)
print(f"  Analysis complete. {len(engine.prompt_log)} decision points recorded.")
print("=" * 60)
print(f"\n  FINAL RECOMMENDATION:\n{final_recommendation}")

## 9. Phase 2 — Extract activations for every recorded prompt

Forward hooks on the chosen layers capture the residual stream at the last token of each prompt.

In [ ]:
write_progress("phase2", 0, 1, "Phase 2 \u00b7 extracting activations...")
print("\n[Phase 2] Extracting activations...")
activation_log = engine.extract_all_prompts(_HOOK_LAYERS)
write_progress("phase2_done", 1, 1, "Phase 2 complete")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n  Captured {len(activation_log)} step activations across layers {_HOOK_LAYERS}")

## 10. Phase 3 — Decode activations through the trained SAE

Pulls a JumpReLU SAE for layer 20 from the HuggingFace Hub (or from `SAE_LOCAL_DIR` if set), encodes each captured activation into a sparse feature vector, and labels the top features. If a local pipeline output dir is provided, also loads the contrastive feature map so each feature carries its training-time theme annotations.

In [ ]:
def _load_contrastive_feature_map(output_dir: str, layer: int) -> dict[int, list[dict]]:
    """Load contrastive_features.json and build a feature_index -> themes mapping."""
    report_path = Path(output_dir) / f"layer_{layer}" / "contrastive_features.json"
    if not report_path.exists():
        return {}

    with open(report_path) as f:
        report = json.load(f)

    feature_map: dict[int, list[dict]] = {}
    for theme, info in report.items():
        if theme.startswith("_"):
            continue
        for feat in info.get("top_features", []):
            idx = feat["feature_index"]
            anchor_act = feat.get("anchor_mean_activation", 0)
            contrast_act = feat.get("contrast_mean_activation", 0)
            direction = "anchor" if anchor_act > contrast_act else "contrast"
            feature_map.setdefault(idx, []).append({
                "theme": theme,
                "rank": feat["rank"],
                "cohens_d": feat["cohens_d"],
                "direction": direction,
            })

    print(f"  Loaded contrastive feature map: {len(feature_map)} features across {len(report) - 1} themes")
    return feature_map


def _load_sae_local(output_dir: str, layer: int, device: str = "cpu"):
    from kiji_inspector.core.sae_core import JumpReLUSAE

    layer_dir = Path(output_dir) / f"layer_{layer}"
    checkpoint = layer_dir / "sae_checkpoints" / "sae_final.pt"
    if not checkpoint.exists():
        print(f"  No local SAE checkpoint at {checkpoint}")
        return None, None

    print(f"  Loading local SAE from {checkpoint}")
    sae = JumpReLUSAE.from_pretrained(str(checkpoint), device=device)
    sae.eval()

    feature_descriptions = None
    desc_path = layer_dir / "activations" / "feature_descriptions.json"
    if desc_path.exists():
        with open(desc_path) as f:
            feature_descriptions = json.load(f)
        print(f"  Loaded {len(feature_descriptions)} feature labels from {desc_path}")

    return sae, feature_descriptions


def _load_sae_from_hub(repo_id: str, layer: int, device: str = "cpu"):
    try:
        sae, feature_descriptions = SAE.from_pretrained(
            repo_id=repo_id,
            layer=layer,
            device=device,
        )
        return sae, feature_descriptions
    except Exception as e:
        print(f"  Could not load SAE from {repo_id} layer {layer}: {e}")
        return None, None


def analyze_activations(
    activation_log: list[tuple[str, dict[str, np.ndarray]]],
    sae_repo_id: str,
    sae_layer: int,
    layer_key: str = "residual_8",
    sae_local_dir: str | None = None,
) -> dict:
    results: dict = {"steps": [], "sae_available": False, "features_available": False}

    # Tier 1: raw activation statistics
    for step_label, acts in activation_log:
        step_info: dict = {"step": step_label, "layers_captured": list(acts.keys()), "raw_stats": {}}
        for layer_name, vec in acts.items():
            step_info["raw_stats"][layer_name] = {
                "mean": float(np.mean(vec)),
                "std": float(np.std(vec)),
                "l2_norm": float(np.linalg.norm(vec)),
                "max_abs": float(np.max(np.abs(vec))),
                "sparsity": float(np.mean(np.abs(vec) < 0.01)),
            }
        results["steps"].append(step_info)

    # Tier 2: SAE feature decomposition (local or HuggingFace Hub)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if sae_local_dir:
        sae, feature_descs = _load_sae_local(sae_local_dir, sae_layer, device=device)
    else:
        sae, feature_descs = _load_sae_from_hub(sae_repo_id, sae_layer, device=device)
    if sae is None:
        print("  SAE not available -- showing raw activation stats only.")
        return results

    # Load contrastive feature map (feature_index -> themes from training)
    contrastive_map: dict[int, list[dict]] = {}
    if sae_local_dir:
        contrastive_map = _load_contrastive_feature_map(sae_local_dir, sae_layer)

    results["sae_available"] = True
    results["contrast_themes"] = list({
        entry["theme"]
        for entries in contrastive_map.values()
        for entry in entries
    }) if contrastive_map else []
    sae.eval()
    sae_dtype = next(sae.parameters()).dtype

    for step_info, (_step_label, acts) in zip(results["steps"], activation_log, strict=True):
        if layer_key not in acts:
            continue
        vec = acts[layer_key]
        vec_tensor = torch.from_numpy(vec).unsqueeze(0).to(device=device, dtype=sae_dtype)
        with torch.no_grad():
            features = sae.encode(vec_tensor)
        features_np = features.squeeze(0).cpu().float().numpy()

        nonzero_mask = features_np > 0
        nonzero_indices = np.where(nonzero_mask)[0]
        nonzero_values = features_np[nonzero_indices]

        sort_order = np.argsort(-nonzero_values)
        top_k = min(20, len(sort_order))
        top_indices = nonzero_indices[sort_order[:top_k]]
        top_values = nonzero_values[sort_order[:top_k]]

        top_features = []
        for idx, val in zip(top_indices, top_values, strict=True):
            feat_entry: dict = {"index": int(idx), "activation": float(val)}
            if int(idx) in contrastive_map:
                feat_entry["themes"] = contrastive_map[int(idx)]
            top_features.append(feat_entry)

        # Aggregate theme scores across all active features for this step
        theme_scores: dict[str, list[float]] = {}
        for idx in nonzero_indices:
            act_val = float(features_np[idx])
            for entry in contrastive_map.get(int(idx), []):
                score = act_val * abs(entry["cohens_d"])
                theme_scores.setdefault(entry["theme"], []).append(score)

        step_info["sae_features"] = {
            "num_active": int(nonzero_mask.sum()),
            "total_features": int(features_np.shape[0]),
            "sparsity_pct": float((1.0 - nonzero_mask.mean()) * 100),
            "top_features": top_features,
            "theme_activations": {
                theme: {
                    "total_score": round(sum(scores), 4),
                    "num_features": len(scores),
                    "mean_score": round(sum(scores) / len(scores), 4),
                }
                for theme, scores in sorted(theme_scores.items(), key=lambda x: -sum(x[1]))
            },
        }

    del sae
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Tier 3: feature label mapping
    if feature_descs is None:
        print("  Feature descriptions not found -- showing SAE features without labels.")
        return results

    results["features_available"] = True
    for step_info in results["steps"]:
        if "sae_features" not in step_info:
            continue
        for feat in step_info["sae_features"]["top_features"]:
            desc = feature_descs.get(str(feat["index"]))
            if desc:
                feat["label"] = desc.get("label", "unknown")
                feat["description"] = desc.get("description", "")
                feat["confidence"] = desc.get("confidence", "low")

    return results


write_progress("phase3", 0, 1, "Phase 3 \u00b7 decoding through SAE...")
print("[Phase 3] Analyzing activations through SAE...")
analysis = analyze_activations(
    activation_log=activation_log,
    sae_repo_id=_SAE_REPO_ID,
    sae_layer=_SAE_LAYER,
    sae_local_dir=SAE_LOCAL_DIR,
)
write_progress("phase3_done", 1, 1, "Phase 3 complete")

## 11. Inspect the SAE features that fired at each decision point

In [ ]:
def print_analysis_summary(analysis: dict):
    print("\n" + "=" * 60)
    print("  ACTIVATION ANALYSIS SUMMARY")
    print("=" * 60)
    print(f"  Decision steps captured: {len(analysis['steps'])}")
    print(f"  SAE available: {analysis['sae_available']}")
    print(f"  Feature labels available: {analysis['features_available']}")

    for step_info in analysis["steps"]:
        print(f"\n  --- {step_info['step']} ---")
        for layer, stats in step_info.get("raw_stats", {}).items():
            if "residual_8" in layer:
                print(f"    L2 norm: {stats['l2_norm']:.2f}, std: {stats['std']:.4f}")
        sae = step_info.get("sae_features")
        if sae:
            print(
                f"    Active features: {sae['num_active']}/{sae['total_features']} "
                f"({100 - sae['sparsity_pct']:.1f}%)"
            )
            for feat in sae["top_features"][:5]:
                label = feat.get("label", f"Feature #{feat['index']}")
                themes = feat.get("themes", [])
                theme_str = ""
                if themes:
                    theme_names = [t["theme"] for t in themes[:2]]
                    theme_str = f"  [{', '.join(theme_names)}]"
                print(f"      {label}: {feat['activation']:.4f}{theme_str}")
            theme_acts = sae.get("theme_activations", {})
            if theme_acts:
                print("    Theme signals:")
                for theme, info in list(theme_acts.items())[:5]:
                    print(f"      {theme}: score={info['total_score']:.2f} ({info['num_features']} features)")


print_analysis_summary(analysis)

## 12. Phase 4 (optional) — Have the model explain its own decisions

Feed the SAE feature trace back into the model and ask for a technical and a plain-language explanation. Heavy on memory — skip if the runtime is tight.

In [ ]:
RUN_EXPLANATIONS = False  # flip to True if you have an A100 with headroom


def render_markdown(text: str, width: int = 100) -> str:
    """Render markdown text for terminal display with table formatting."""
    lines = text.split("\n")
    output = []
    table_buffer: list[str] = []
    in_table = False

    def flush_table():
        if not table_buffer:
            return
        rows = []
        for line in table_buffer:
            cells = [c.strip() for c in line.strip().strip("|").split("|")]
            rows.append(cells)
        if len(rows) < 2:
            output.extend(table_buffer)
            return
        header = rows[0]
        data_rows = [r for r in rows[1:] if not all(set(c.strip()) <= {"-", ":"} for c in r)]
        n_cols = len(header)
        col_widths = [len(h) for h in header]
        for row in data_rows:
            for i, cell in enumerate(row[:n_cols]):
                col_widths[i] = max(col_widths[i], len(cell))
        col_widths = [min(w, 50) for w in col_widths]

        def make_row(cells, widths):
            parts = []
            for cell, w in zip(cells, widths, strict=False):
                if len(cell) > w:
                    cell = cell[: w - 3] + "..."
                parts.append(cell.ljust(w))
            return "| " + " | ".join(parts) + " |"

        def make_separator(widths):
            return "+-" + "-+-".join("-" * w for w in widths) + "-+"

        output.append(make_separator(col_widths))
        output.append(make_row(header, col_widths))
        output.append(make_separator(col_widths))
        for row in data_rows:
            row = row + [""] * (n_cols - len(row))
            output.append(make_row(row, col_widths))
        output.append(make_separator(col_widths))

    for line in lines:
        stripped = line.strip()
        if stripped.startswith("|") and "|" in stripped[1:]:
            if not in_table:
                in_table = True
                table_buffer = []
            table_buffer.append(line)
        else:
            if in_table:
                flush_table()
                table_buffer = []
                in_table = False
            if stripped.startswith("**") and stripped.endswith("**"):
                output.extend(["", stripped, ""])
            elif stripped.startswith("* ") or stripped.startswith("- "):
                output.append(textwrap.fill(stripped, width=width, subsequent_indent="  "))
            elif stripped:
                output.append(textwrap.fill(stripped, width=width))
            else:
                output.append("")

    if in_table:
        flush_table()

    return "\n".join(output)


def _build_feature_summary(analysis_results: dict) -> str:
    lines = []
    for step_info in analysis_results["steps"]:
        lines.append(f"## {step_info['step']}")
        for layer, stats in step_info.get("raw_stats", {}).items():
            if "residual_8" in layer:
                lines.append(
                    f"  L2={stats['l2_norm']:.2f}, mean={stats['mean']:.4f}, std={stats['std']:.4f}"
                )
        sae = step_info.get("sae_features")
        if sae:
            lines.append(
                f"  Active SAE features: {sae['num_active']}/{sae['total_features']} "
                f"({100 - sae['sparsity_pct']:.1f}% active)"
            )
            for feat in sae["top_features"][:5]:
                label = feat.get("label", f"Feature #{feat['index']}")
                desc = feat.get("description", "")
                line = f"    - {label} (activation={feat['activation']:.4f})"
                if desc:
                    line += f": {desc}"
                lines.append(line)
        lines.append("")
    return "\n".join(lines)


def generate_decision_explanations(engine, analysis_results, agent_output):
    feature_summary = _build_feature_summary(analysis_results)

    technical_prompt = engine._build_prompt(
        (
            "You are an AI interpretability researcher. You have access to "
            "Sparse Autoencoder (SAE) analysis of a home repair AI advisor's "
            "internal activations captured at each decision step. Explain what "
            "the activation patterns reveal about HOW the AI made its decisions."
        ),
        (
            "A home repair AI advisor analyzed three household problems "
            "(dishwasher leak, stuck garbage disposal, noisy water heater). "
            "Here is its final recommendation:\n\n"
            f"{agent_output[:2000]}\n\n"
            "Here is the SAE feature analysis at each step "
            "(steps are labeled as PROBLEM_ToolName):\n\n"
            f"{feature_summary}\n\n"
            "Please explain:\n"
            "1. What patterns in the agent's internal representations drove its decisions?\n"
            "2. How did the active features change across problem types and tool types?\n"
            "3. What does this reveal about how the agent distinguishes DIY-safe repairs from those requiring professionals?"
        ),
    )
    technical = engine.generate(technical_prompt, "explain_technical", max_tokens=1024)

    feature_evidence_lines = []
    for step_info in analysis_results["steps"]:
        sae = step_info.get("sae_features")
        if not sae:
            continue
        top3 = sae["top_features"][:3]
        labeled = [f for f in top3 if f.get("label")]
        if not labeled:
            continue
        parts = ", ".join(f'"{f["label"]}" (strength {f["activation"]:.1f})' for f in labeled)
        step = step_info["step"]
        step_readable = step.replace("_", " ").lower()
        if "final" in step_readable:
            step_readable = "making the final recommendation"
        else:
            step_readable = f"analyzing {step_readable}"
        feature_evidence_lines.append(
            f"- While {step_readable}: strongest brain signals were {parts}"
        )
    feature_evidence = "\n".join(feature_evidence_lines) or "- No labeled features were available."

    layman_prompt = engine._build_prompt(
        (
            "You explain AI decisions in plain, everyday language. No jargon, "
            "no technical terms. Write as if explaining to someone with no "
            "technical background."
        ),
        (
            "An AI home repair advisor was asked to help with three problems: "
            "a leaking dishwasher, a stuck garbage disposal, and a noisy water "
            "heater. Here is what it recommended:\n\n"
            f"{agent_output[:1500]}\n\n"
            "We looked inside the AI's brain at each step. Here is what we found:\n\n"
            f"{feature_evidence}\n\n"
            "Using the brain signals above, explain in 4-6 plain sentences "
            "what was going through the AI's mind when making these "
            "recommendations. Reference which signals were strongest and "
            "how they differed between the three repair problems. "
            "Start with 'The AI advisor decided'."
        ),
    )
    layman = engine.generate(layman_prompt, "explain_layman", max_tokens=512)
    return technical, layman


technical, layman = "", ""
if RUN_EXPLANATIONS and analysis["sae_available"]:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("\n[Phase 4] Generating decision explanations...")
    technical, layman = generate_decision_explanations(engine, analysis, final_recommendation)
    print("\n" + "=" * 60)
    print("  TECHNICAL EXPLANATION")
    print("=" * 60)
    print(render_markdown(technical))
    print("\n" + "=" * 60)
    print("  PLAIN LANGUAGE SUMMARY")
    print("=" * 60)
    print(render_markdown(layman))
else:
    print("Skipping Phase 4. Set RUN_EXPLANATIONS = True to enable.")

## 13. Release the model

In [ ]:
engine.cleanup()

## 14. Phase 5 — Save raw results to disk

In [ ]:
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "analysis_results.json", "w") as f:
    json.dump(analysis, f, indent=2, default=str)
with open(OUTPUT_DIR / "agent_output.txt", "w") as f:
    f.write(final_recommendation)
with open(OUTPUT_DIR / "per_problem_analyses.json", "w") as f:
    json.dump(per_problem, f, indent=2)
if technical:
    (OUTPUT_DIR / "technical_explanation.txt").write_text(technical)
if layman:
    (OUTPUT_DIR / "layman_explanation.txt").write_text(layman)

print(f"Saved raw results to {OUTPUT_DIR}/")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)

## 15. Phase 6 — Build `ui_data.json` for the static viewer

Transforms the raw analysis into the shape `demo/home_repair/index.html` expects: per-problem cards, tool-result HTML snippets, the most *distinctive* SAE features for each (problem, tool) pair, recommendations, theme comparison scores, and a description block driven by `home_repair.json`.

In [ ]:
_TOOL_DISPLAY_NAMES = {
    "ManualCheck": "Repair Manual Lookup",
    "PartsSearch": "Parts & Pricing Search",
    "TutorialSearch": "Video Tutorial Search",
    "ProQuote": "Professional Quote",
}

_PROBLEM_META = {
    "dishwasher_leak": {
        "icon": "\U0001f4a7",
        "urgency": {"label": "Fix Soon", "level": "yellow"},
        "difficulty": {"label": "Moderate", "level": "yellow"},
        "costRange": "$13\u2013$90 DIY",
    },
    "disposal_stuck": {
        "icon": "\u2699\ufe0f",
        "urgency": {"label": "Can Wait", "level": "green"},
        "difficulty": {"label": "Easy", "level": "green"},
        "costRange": "$0\u2013$8 DIY",
    },
    "water_heater_noise": {
        "icon": "\U0001f525",
        "urgency": {"label": "Act Now", "level": "red"},
        "difficulty": {"label": "Difficult", "level": "red"},
        "costRange": "$15\u2013$649 DIY",
    },
}

_THEME_KEYWORDS = {
    "Safety Concern": ["safety", "hazard", "gas", "risk", "danger"],
    "Cost Sensitivity": ["cost", "budget", "price", "replacement cost", "expense"],
    "DIY Feasibility": ["diy", "beginner", "skill", "tool requirement"],
    "Urgency Level": ["urgent", "immediate", "damage", "active"],
    "Age / Warranty Factor": ["age", "lifespan", "warranty", "coverage"],
}


def _summarize_manual(data: dict) -> str:
    causes = ", ".join(c.split(" -- ")[0] for c in data.get("possible_causes", []))
    safety = data.get("safety", "")
    difficulty = data.get("diy_difficulty", "")
    parts = [f"<strong>Possible causes:</strong> {causes}"]
    if difficulty:
        parts.append(f"<strong>DIY difficulty:</strong> {difficulty}")
    if safety:
        parts.append(f"<strong>Safety:</strong> {safety}")
    if data.get("quick_fix"):
        parts.append(f"<strong>Quick fix:</strong> {data['quick_fix']}")
    return "<br>".join(parts)


def _summarize_parts(data: dict) -> str:
    items = [f"<strong>{p['name']}:</strong> ${p['price']:.2f}" for p in data.get("parts", [])[:3]]
    line1 = " &middot; ".join(items)
    line2 = data.get("diy_cost_range", "")
    return f"{line1}<br>{line2}" if line2 else line1


def _summarize_tutorials(data: dict) -> str:
    lines = []
    for r in data.get("results", [])[:2]:
        meta = []
        if r.get("views"):
            meta.append(f"{r['views']} views")
        if r.get("duration"):
            meta.append(r["duration"])
        if r.get("difficulty"):
            meta.append(r["difficulty"])
        meta_str = f" ({', '.join(meta)})" if meta else ""
        channel = r.get("channel", "")
        lines.append(f"<strong>\"{r['title']}\"</strong> by {channel}{meta_str}")
    return "<br>".join(lines)


def _summarize_pro_quote(data: dict) -> str:
    items = [f"<strong>{est['repair']}:</strong> ${est['total']} total" for est in data.get("repair_estimates", [])]
    line1 = " &middot; ".join(items)
    extras = []
    if data.get("diagnosis_fee"):
        extras.append(f"<strong>Diagnosis fee:</strong> ${data['diagnosis_fee']}")
    if data.get("next_available"):
        extras.append(f"Available in {data['next_available']}")
    if data.get("warranty_on_repair"):
        extras.append(f"{data['warranty_on_repair']} warranty")
    line2 = ". ".join(extras)
    return f"{line1}<br>{line2}"


_TOOL_SUMMARIZERS = {
    "ManualCheck": _summarize_manual,
    "PartsSearch": _summarize_parts,
    "TutorialSearch": _summarize_tutorials,
    "ProQuote": _summarize_pro_quote,
}


def _generate_feature_sentence(tool_name: str, features: list[dict]) -> str:
    if not features:
        return ""
    display = _TOOL_DISPLAY_NAMES.get(tool_name, tool_name)
    top = features[:2]
    labels = [f"<strong>{f['label']}</strong>" for f in top]
    focus = f"{labels[0]} and {labels[1]}" if len(labels) == 2 else labels[0]
    verb = {
        "ManualCheck": "checking the repair manual",
        "PartsSearch": "reviewing parts and pricing",
        "TutorialSearch": "searching for tutorials",
        "ProQuote": "reviewing professional quotes",
    }.get(tool_name, f"running {display}")
    return f"While {verb}, the AI focused most on {focus}."


def _derive_comparison_scores(
    sae_features: dict[str, dict[str, dict]],
) -> dict[str, dict[str, int]]:
    comparison: dict[str, dict[str, float]] = {theme: {} for theme in _THEME_KEYWORDS}
    for pid in sae_features:
        all_features: list[dict] = []
        for tool_data in sae_features[pid].values():
            all_features.extend(tool_data.get("features", []))

        for theme, keywords in _THEME_KEYWORDS.items():
            score = 0.0
            count = 0
            for f in all_features:
                label_lower = f.get("label", "").lower()
                if any(kw in label_lower for kw in keywords):
                    score += f.get("strength", 0)
                    count += 1
            comparison[theme][pid] = round(score / max(count, 1) * 100) if count else 10
    return comparison


def build_ui_data(
    analysis: dict,
    per_problem: dict[str, str],
    final_recommendation: str,
    home_repair_json_path: Path,
) -> dict:
    # problems
    problems = []
    for p in _PROBLEMS:
        meta = _PROBLEM_META.get(p["id"], {})
        problems.append({
            "id": p["id"],
            "icon": meta.get("icon", ""),
            "title": p["summary"],
            "appliance": p["appliance"],
            "age": p["age"],
            "details": p["details"],
            "urgency": meta.get("urgency", {"label": "Unknown", "level": "yellow"}),
            "difficulty": meta.get("difficulty", {"label": "Unknown", "level": "yellow"}),
            "costRange": meta.get("costRange", ""),
        })

    # toolResults
    tool_results: dict[str, dict[str, str]] = {}
    for p in _PROBLEMS:
        pid = p["id"]
        tool_results[pid] = {}
        for tool_name, (source, _) in _TOOLS.items():
            data = source.get(pid, {})
            summarizer = _TOOL_SUMMARIZERS.get(tool_name)
            tool_results[pid][tool_name] = summarizer(data) if summarizer else json.dumps(data)

    # saeFeatures (ranked by deviation from cross-step mean)
    sae_features: dict[str, dict[str, dict]] = {}
    step_lookup: dict[str, dict] = {step_info["step"]: step_info for step_info in analysis.get("steps", [])}

    feature_activations: dict[int, list[float]] = {}
    for step_info in analysis.get("steps", []):
        if "sae_features" not in step_info:
            continue
        for f in step_info["sae_features"].get("top_features", []):
            feature_activations.setdefault(f.get("index", -1), []).append(f.get("activation", 0))

    feature_mean: dict[int, float] = {
        idx: sum(vals) / len(vals) for idx, vals in feature_activations.items()
    }

    for p in _PROBLEMS:
        pid = p["id"]
        sae_features[pid] = {}
        for tool_name in _TOOLS:
            step_key = f"{pid}_{tool_name}"
            step_info = step_lookup.get(step_key)

            features_list: list[dict] = []
            if step_info and "sae_features" in step_info:
                top_feats = step_info["sae_features"].get("top_features", [])
                scored = []
                for f in top_feats:
                    idx = f.get("index", -1)
                    act = f.get("activation", 0)
                    mean = feature_mean.get(idx, act)
                    scored.append((act - mean, f))
                scored.sort(key=lambda x: x[0], reverse=True)

                distinctive = scored[:5]
                max_dev = max((abs(d) for d, _ in distinctive), default=1.0) or 1.0
                for dev, f in distinctive:
                    features_list.append({
                        "label": f.get("label", f"Feature #{f.get('index', '?')}"),
                        "strength": round(abs(dev) / max_dev, 2),
                        "description": f.get("description", ""),
                    })

            sentence = _generate_feature_sentence(tool_name, features_list)
            sae_features[pid][tool_name] = {"features": features_list, "sentence": sentence}

    # recommendations
    pro_quote_data = _TOOLS["ProQuote"][0]
    parts_data = _TOOLS["PartsSearch"][0]

    recommendations: dict[str, dict] = {}
    for p in _PROBLEMS:
        pid = p["id"]
        meta = _PROBLEM_META.get(pid, {})
        pq = pro_quote_data.get(pid, {})
        pd_parts = parts_data.get(pid, {})

        difficulty_level = meta.get("difficulty", {}).get("level", "yellow")
        if difficulty_level == "red":
            verdict, verdict_label = "pro", "Call a Professional"
        elif difficulty_level == "green":
            verdict, verdict_label = "diy", "Easy DIY Fix"
        else:
            verdict, verdict_label = "diy", "DIY Repair"

        diy_cost = pd_parts.get("diy_cost_range", "")
        estimates = pq.get("repair_estimates", [])
        if estimates:
            lo = min(e["total"] for e in estimates)
            hi = max(e["total"] for e in estimates)
            pro_cost = f"${lo}\u2013${hi}" if lo != hi else f"${lo}"
        else:
            pro_cost = ""

        raw = per_problem.get(pid, "")
        sections = raw.split("[ProQuote]")
        rationale = sections[-1].strip() if len(sections) > 1 else raw.strip()
        if len(rationale) > 500:
            rationale = rationale[:497] + "..."

        recommendations[pid] = {
            "verdict": verdict,
            "verdictLabel": verdict_label,
            "diyCost": diy_cost,
            "proCost": pro_cost,
            "rationale": rationale,
        }

    # comparison
    comparison = _derive_comparison_scores(sae_features)

    # themes
    themes_raw = {
        "diy_vs_professional": {"title": "DIY vs. Professional", "leftLabel": "Easy DIY", "rightLabel": "Needs a Pro"},
        "urgent_vs_planned": {"title": "Urgent vs. Planned", "leftLabel": "Can Wait", "rightLabel": "Act Now"},
        "cheap_fix_vs_replacement": {"title": "Cheap Fix vs. Replacement", "leftLabel": "Quick Part Swap", "rightLabel": "Consider Replacing"},
        "safe_vs_hazardous": {"title": "Safe vs. Hazardous", "leftLabel": "Low Risk", "rightLabel": "High Hazard"},
        "warranty_vs_out_of_pocket": {"title": "Warranty vs. Out of Pocket", "leftLabel": "May Be Covered", "rightLabel": "Out of Pocket"},
    }

    contrast_descriptions = {}
    if home_repair_json_path.exists():
        with open(home_repair_json_path) as f:
            hr_config = json.load(f)
        contrast_descriptions = hr_config.get("contrast_types", {})

    theme_to_comparison = {
        "diy_vs_professional": "DIY Feasibility",
        "urgent_vs_planned": "Urgency Level",
        "cheap_fix_vs_replacement": "Cost Sensitivity",
        "safe_vs_hazardous": "Safety Concern",
        "warranty_vs_out_of_pocket": "Age / Warranty Factor",
    }

    themes = []
    for theme_id, tmeta in themes_raw.items():
        desc = contrast_descriptions.get(theme_id, "")
        comp_key = theme_to_comparison.get(theme_id, "")
        comp_scores = comparison.get(comp_key, {})
        markers = {pid: comp_scores.get(pid, 50) for pid in ["dishwasher_leak", "disposal_stuck", "water_heater_noise"]}

        comp_theme_kws = _THEME_KEYWORDS.get(comp_key, [])
        driving_features: list[str] = []
        for pid_data in sae_features.values():
            for tool_data in pid_data.values():
                for f in tool_data.get("features", []):
                    lbl = f.get("label", "").lower()
                    if any(kw in lbl for kw in comp_theme_kws) and f["label"] not in driving_features:
                        driving_features.append(f["label"])
        driving_features = driving_features[:2]
        if driving_features:
            features_html = "Driven by " + " and ".join(f"<strong>{lbl}</strong>" for lbl in driving_features)
        else:
            features_html = ""

        themes.append({
            "id": theme_id,
            "title": tmeta["title"],
            "description": desc,
            "leftLabel": tmeta["leftLabel"],
            "rightLabel": tmeta["rightLabel"],
            "markers": markers,
            "features": features_html,
        })

    return {
        "problems": problems,
        "toolResults": tool_results,
        "saeFeatures": sae_features,
        "recommendations": recommendations,
        "comparison": comparison,
        "themes": themes,
    }


print("\n[Phase 6] Generating UI data...")
ui_data = build_ui_data(
    analysis=analysis,
    per_problem=per_problem,
    final_recommendation=final_recommendation,
    home_repair_json_path=HOME_REPAIR_JSON,
)
with open(OUTPUT_DIR / "ui_data.json", "w") as f:
    json.dump(ui_data, f, indent=2, ensure_ascii=False)
write_progress("complete", 1, 1, "Done \u2014 live results loaded.")

print(f"\n  Results saved to {OUTPUT_DIR}/")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)
print("\n  Download ui_data.json and drop it next to demo/home_repair/index.html to view the interactive explanation.")

## 16. Re-open the live viewer (optional)

The viewer was already started in section 8a and the results land in `output/ui_data.json` automatically. Run this cell only if you closed the tab and want to re-open it.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(PORT)